# Serve the fine-tuned ScholarPulse model (Kaggle T4)

Serves Qwen2.5-1.5B-Instruct + the QLoRA adapter from `finetune_qlora.ipynb` as an
**OpenAI-compatible API** (model name `scholarpulse`), then exposes it via a Cloudflare tunnel
so the ScholarPulse backend can reach it as `FINETUNED_BASE_URL`.

**Settings:** Accelerator → **GPU T4**, **Internet ON**. Then **Run All**.

1. Adapter: add `qlora_adapter.zip` as a Kaggle dataset (**Add Input**), or use the upload widget below.
2. vLLM serves the base model + LoRA. If vLLM fails, a small transformers + peft server takes over.
3. The big URL printed by the tunnel cell goes into `.env` as `FINETUNED_BASE_URL=<url>/v1`.

🔑 A random API key is generated at the start; both servers reject requests without
`Authorization: Bearer <key>`. Put it in `.env` as `FINETUNED_API_KEY`. The URL itself is still public,
so don't share the key and stop the session when you are done.

In [ ]:
import glob, io, os, re, secrets, shutil, subprocess, sys, time, zipfile
from pathlib import Path

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_NAME = "scholarpulse"                      # = FINETUNED_MODEL in the backend .env
ADAPTER_DIR = Path("/kaggle/working/qlora_adapter")
PORT = 8000
LOCAL_URL = f"http://localhost:{PORT}"
LOG_DIR = Path("/kaggle/working/logs"); LOG_DIR.mkdir(parents=True, exist_ok=True)
API_KEY = secrets.token_urlsafe(24)             # = FINETUNED_API_KEY in the backend .env (new every run)
AUTH_HEADER = {"Authorization": f"Bearer {API_KEY}"}
print("API key generated (printed again with the tunnel URL).")

## 1. Upload the adapter

Uses `qlora_adapter.zip` (or an already-extracted adapter) from `/kaggle/input` if you attached it as a dataset.
Otherwise it shows an upload widget: pick the zip, wait for the upload to finish, then run the next cell.

In [ ]:
def find_adapter_source():
    """A zip, or a directory that holds adapter_config.json, from /kaggle/input or /kaggle/working."""
    for pattern in ["/kaggle/input/**/qlora_adapter*.zip", "/kaggle/working/qlora_adapter*.zip"]:
        hits = glob.glob(pattern, recursive=True)
        if hits:
            return Path(hits[0])
    # Kaggle extracts uploaded zips into datasets automatically.
    hits = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    return Path(hits[0]).parent if hits else None

SOURCE = find_adapter_source()
upload = None
if SOURCE:
    print("Found adapter source:", SOURCE)
else:
    import ipywidgets as widgets
    from IPython.display import display
    upload = widgets.FileUpload(accept=".zip", multiple=False, description="qlora_adapter.zip")
    display(upload)
    print("Select qlora_adapter.zip, wait until the button shows (1), then run the next cell.")

In [ ]:
def uploaded_bytes(widget):
    """File bytes from an ipywidgets FileUpload (v7 dict or v8 tuple format)."""
    value = widget.value
    item = next(iter(value.values())) if isinstance(value, dict) else value[0]
    return bytes(item["content"])

if ADAPTER_DIR.exists():
    shutil.rmtree(ADAPTER_DIR)
tmp = Path("/kaggle/working/_adapter_unzip")
shutil.rmtree(tmp, ignore_errors=True)

if SOURCE is not None and SOURCE.is_dir():
    shutil.copytree(SOURCE, tmp)
elif SOURCE is not None:
    zipfile.ZipFile(SOURCE).extractall(tmp)
elif upload is not None and upload.value:
    zipfile.ZipFile(io.BytesIO(uploaded_bytes(upload))).extractall(tmp)
else:
    raise RuntimeError("No adapter: attach qlora_adapter.zip as a dataset or upload it in the cell above.")

# The zip normally has files at its root; also handle a nested qlora_adapter/ folder.
config = next(tmp.rglob("adapter_config.json"), None)
if config is None:
    raise RuntimeError("adapter_config.json not found in the zip; is this the QLoRA adapter?")
shutil.move(str(config.parent), ADAPTER_DIR)
shutil.rmtree(tmp, ignore_errors=True)
print("Adapter ready at", ADAPTER_DIR, "->", sorted(p.name for p in ADAPTER_DIR.iterdir()))

## 2. vLLM server (base model + LoRA)

T4 has no bf16, hence `--dtype half`. It starts with a 32k context; if that fails with a memory or
length error, it retries once with 8k before using the fallback. The first start downloads the model and
can take a few minutes.

In [ ]:
import urllib.request

def wait_for_server(proc=None, timeout_s=900, url=f"{LOCAL_URL}/v1/models"):
    """Poll /v1/models (with the API key) until it answers. Returns False if proc dies or the timeout passes."""
    start = time.time()
    while time.time() - start < timeout_s:
        if proc is not None and proc.poll() is not None:
            print(f"Server process exited with code {proc.returncode}")
            return False
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=AUTH_HEADER), timeout=5) as r:
                print(f"Server up after {time.time() - start:.0f}s:", r.read().decode()[:300])
                return True
        except Exception:
            time.sleep(10)
    print("Timed out waiting for", url)
    return False

def tail(path, n=30):
    lines = Path(path).read_text(errors="replace").splitlines()
    print("\n".join(lines[-n:]))

SERVER_UP, SERVER_KIND, server_proc = False, None, None
pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"],
                     capture_output=True, text=True)
VLLM_OK = pip.returncode == 0
print("vllm install:", "ok" if VLLM_OK else "FAILED\n" + pip.stderr[-2000:])

In [ ]:
# Startup errors worth retrying with a shorter context (KV cache too small / OOM / length checks).
MEM_OR_LEN_ERROR = re.compile(
    r"out of memory|OutOfMemory|KV cache|max_model_len|max seq len|maximum number of tokens|max-model-len",
    re.I)

def start_vllm(max_model_len):
    """Start vLLM; returns (process or None, log path). A failed process is killed to free the GPU."""
    log_path = LOG_DIR / f"vllm_{max_model_len}.log"
    proc = subprocess.Popen(
        ["vllm", "serve", BASE_MODEL,
         "--dtype", "half",
         "--enable-lora",
         "--lora-modules", f"{LORA_NAME}={ADAPTER_DIR}",
         "--max-model-len", str(max_model_len),
         "--gpu-memory-utilization", "0.92",
         "--api-key", API_KEY,
         "--port", str(PORT)],
        stdout=open(log_path, "w"), stderr=subprocess.STDOUT,
    )
    if wait_for_server(proc):
        return proc, log_path
    tail(log_path)
    proc.kill(); proc.wait()
    return None, log_path

if VLLM_OK:
    server_proc, log_path = start_vllm(32768)
    if server_proc is None and MEM_OR_LEN_ERROR.search(log_path.read_text(errors="replace")):
        print("\nMemory/length error at 32768 tokens; retrying once with --max-model-len 8192\n")
        server_proc, log_path = start_vllm(8192)
    SERVER_UP = server_proc is not None
    if SERVER_UP:
        SERVER_KIND = "vllm"
print("SERVER_UP =", SERVER_UP, "| kind =", SERVER_KIND)

## 3. Fallback: transformers + peft server (only runs if vLLM failed)

A minimal FastAPI app that serves `POST /v1/chat/completions` (and `GET /v1/models`) in the OpenAI
format, in fp16 on the GPU, and rejects requests without `Authorization: Bearer <API_KEY>`.
It's slower than vLLM and handles one request at a time.

In [ ]:
FALLBACK_SERVER = r"""
import hmac, os, threading, time, uuid
import torch
from fastapi import Depends, FastAPI, Header, HTTPException
from peft import PeftModel
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL, ADAPTER_DIR, LORA_NAME = "%(base)s", "%(adapter)s", "%(name)s"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="cuda")
model = PeftModel.from_pretrained(model, ADAPTER_DIR).merge_and_unload().eval()
lock = threading.Lock()
API_KEY = os.environ["SERVE_API_KEY"]


def require_key(authorization: str | None = Header(default=None)) -> None:
    token = (authorization or "").removeprefix("Bearer ").strip()
    if not authorization or not authorization.startswith("Bearer ") or not hmac.compare_digest(token, API_KEY):
        raise HTTPException(status_code=401, detail="Missing or invalid API key")


app = FastAPI(dependencies=[Depends(require_key)])


class Message(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    model: str = LORA_NAME
    messages: list[Message]
    max_tokens: int = 1024
    temperature: float = 0.0


@app.get("/v1/models")
def models():
    return {"object": "list", "data": [{"id": LORA_NAME, "object": "model", "owned_by": "scholarpulse"}]}


@app.post("/v1/chat/completions")
def chat(req: ChatRequest):
    prompt = tokenizer.apply_chat_template([m.model_dump() for m in req.messages],
                                           tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to("cuda")
    sample = req.temperature > 0
    with lock, torch.inference_mode():
        out = model.generate(**enc, max_new_tokens=req.max_tokens, do_sample=sample,
                             temperature=req.temperature if sample else None, top_p=None, top_k=None,
                             pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    new = out[0, enc["input_ids"].shape[1]:]
    text = tokenizer.decode(new, skip_special_tokens=True).strip()
    n_in, n_out = enc["input_ids"].shape[1], new.shape[0]
    return {
        "id": f"chatcmpl-{uuid.uuid4().hex}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": text},
                     "finish_reason": "length" if n_out >= req.max_tokens else "stop"}],
        "usage": {"prompt_tokens": n_in, "completion_tokens": int(n_out), "total_tokens": n_in + int(n_out)},
    }
""" % {"base": BASE_MODEL, "adapter": ADAPTER_DIR, "name": LORA_NAME}

if not SERVER_UP:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "fastapi", "uvicorn", "accelerate"],
                   check=True)
    Path("/kaggle/working/fallback_server.py").write_text(FALLBACK_SERVER)
    fb_log = open(LOG_DIR / "fallback.log", "w")
    server_proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "fallback_server:app", "--host", "0.0.0.0", "--port", str(PORT)],
        cwd="/kaggle/working", stdout=fb_log, stderr=subprocess.STDOUT,
        env={**os.environ, "SERVE_API_KEY": API_KEY},  # passed via env, not written into the script
    )
    SERVER_UP = wait_for_server(server_proc, timeout_s=600)
    if SERVER_UP:
        SERVER_KIND = "transformers+peft"
    else:
        tail(LOG_DIR / "fallback.log")
        raise RuntimeError("Fallback server failed to start; see the log above.")
print("SERVER_UP =", SERVER_UP, "| kind =", SERVER_KIND)

## 4. Cloudflare tunnel

In [ ]:
from IPython.display import HTML, display

CLOUDFLARED = Path("/kaggle/working/cloudflared")
if not CLOUDFLARED.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CLOUDFLARED)
    CLOUDFLARED.chmod(0o755)

tunnel_log_path = LOG_DIR / "cloudflared.log"
tunnel_proc = subprocess.Popen([str(CLOUDFLARED), "tunnel", "--url", LOCAL_URL, "--no-autoupdate"],
                               stdout=open(tunnel_log_path, "w"), stderr=subprocess.STDOUT)

PUBLIC_URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel_log_path.read_text(errors="replace"))
    if m:
        PUBLIC_URL = m.group(0)
        break
    time.sleep(2)
if PUBLIC_URL is None:
    tail(tunnel_log_path)
    raise RuntimeError("No trycloudflare URL found; see the log above.")

display(HTML(
    f"<div style='font-size:34px;font-weight:700;padding:16px 0;word-break:break-all'>{PUBLIC_URL}</div>"
    f"<div style='font-size:18px'>Backend <code>.env</code>:<br>"
    f"<code>FINETUNED_BASE_URL={PUBLIC_URL}/v1</code><br><code>FINETUNED_MODEL={LORA_NAME}</code><br>"
    f"<code>FINETUNED_API_KEY={API_KEY}</code></div>"
))
print(f"\nFINETUNED_BASE_URL={PUBLIC_URL}/v1\nFINETUNED_MODEL={LORA_NAME}\nFINETUNED_API_KEY={API_KEY}")

## 5. Test through the public URL (OpenAI client)

The DNS for a new trycloudflare host can take a few seconds to resolve, so this retries.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)
import openai
from openai import OpenAI

client = OpenAI(base_url=f"{PUBLIC_URL}/v1", api_key=API_KEY, timeout=60)
messages = [  # same format the adapter was trained on
    {"role": "system", "content": "You are a research assistant. Cite sources as [Doc N, Sec S]."},
    {"role": "user", "content": "Summarize this paper in one sentence (TLDR) and cite the section you used.\n\n"
        "[Doc 1, Sec 1] We propose the Transformer, a network architecture based solely on attention "
        "mechanisms, dispensing with recurrence and convolutions entirely.\n\n"
        "[Doc 1, Sec 2] Experiments on two machine translation tasks show these models are superior in "
        "quality while being more parallelizable and requiring significantly less time to train."},
]
for attempt in range(6):
    try:
        t0 = time.time()
        resp = client.chat.completions.create(model=LORA_NAME, messages=messages, max_tokens=300, temperature=0)
        print(f"[{SERVER_KIND}] {time.time() - t0:.1f}s ->", resp.choices[0].message.content)
        break
    except openai.AuthenticationError as exc:
        raise RuntimeError("Server rejected API_KEY; is it the key printed above?") from exc
    except Exception as exc:
        print(f"attempt {attempt + 1} failed: {exc}")
        time.sleep(10)

## 6. Keep the session alive

Leave this running while the backend uses the endpoint. Interrupt it to stop (servers keep running until the session ends).

In [ ]:
while True:
    alive = {"server": server_proc.poll() is None, "tunnel": tunnel_proc.poll() is None}
    print(time.strftime("%H:%M:%S"), PUBLIC_URL, alive, flush=True)
    if not all(alive.values()):
        print("A process died; check", LOG_DIR)
    time.sleep(300)